# 03 - Eletronica e Instrumentacao
> Design de circuitos, analise de sinais e programacao de microcontroladores

**Modulo:** `11_engfis_tecnico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
# === Setup Padrao ===
import os
from anthropic import Anthropic
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal

client = Anthropic()  # usa ANTHROPIC_API_KEY do ambiente
MODEL = "claude-sonnet-4-20250514"

def ask_claude(prompt: str, system: str = "Voce e um engenheiro eletronico e de instrumentacao especialista. Responda em portugues sem acentos.") -> str:
    """Envia prompt ao Claude e retorna resposta."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

print("Setup completo! Bibliotecas carregadas.")

## Design de Circuitos com Claude

Vamos pedir ao Claude para projetar um circuito de condicionamento de sinal
para um sensor de temperatura do tipo RTD (PT100). Isso inclui:

- **Ponte de Wheatstone** para converter variacao de resistencia em tensao
- **Amplificador de instrumentacao** para amplificar o sinal diferencial
- **Calculo de componentes** para a faixa de temperatura desejada

In [ ]:
# Pedir ao Claude para projetar o circuito de condicionamento
prompt_circuito = """
Projete um circuito de condicionamento de sinal para um sensor RTD PT100
com as seguintes especificacoes:

- Faixa de temperatura: 0 a 200 graus Celsius
- Saida desejada: 0 a 5V (compativel com ADC de microcontrolador)
- Alimentacao disponivel: 5V e 3.3V

Inclua:
1. Descricao do esquematico (ponte de Wheatstone + amplificador de instrumentacao)
2. Calculo de todos os valores de componentes (resistores, ganho do amplificador)
3. Equacoes de transferencia do circuito
4. Consideracoes sobre linearizacao e precisao

Use a equacao do PT100: R(T) = R0 * (1 + alpha*T) com R0=100 ohms e alpha=0.00385/C
"""

resposta_circuito = ask_claude(prompt_circuito)
print(resposta_circuito)

In [ ]:
# Visualizar a curva de resposta do PT100 e o circuito proposto
T = np.linspace(0, 200, 500)  # Temperatura em Celsius
R0 = 100.0   # Resistencia a 0C em ohms
alpha = 0.00385  # Coeficiente de temperatura

# Resistencia do PT100 em funcao da temperatura
R_pt100 = R0 * (1 + alpha * T)

# Ponte de Wheatstone: R1=R2=R3=100 ohms, Rx=PT100
R1 = 100.0
R2 = 100.0
R3 = 100.0
V_exc = 5.0  # Tensao de excitacao

# Tensao diferencial da ponte
V_bridge = V_exc * (R_pt100 / (R3 + R_pt100) - R2 / (R1 + R2))

# Amplificador de instrumentacao - ganho para mapear para 0-5V
V_bridge_max = V_exc * (R0*(1+alpha*200) / (R3 + R0*(1+alpha*200)) - R2/(R1+R2))
Ganho_InAmp = 5.0 / V_bridge_max
V_saida = Ganho_InAmp * V_bridge

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(T, R_pt100, 'b-', linewidth=2)
axes[0].set_xlabel('Temperatura (C)')
axes[0].set_ylabel('Resistencia (ohms)')
axes[0].set_title('Curva do PT100')
axes[0].grid(True, alpha=0.3)

axes[1].plot(T, V_bridge * 1000, 'r-', linewidth=2)
axes[1].set_xlabel('Temperatura (C)')
axes[1].set_ylabel('Tensao Diferencial (mV)')
axes[1].set_title('Saida da Ponte de Wheatstone')
axes[1].grid(True, alpha=0.3)

axes[2].plot(T, V_saida, 'g-', linewidth=2)
axes[2].set_xlabel('Temperatura (C)')
axes[2].set_ylabel('Tensao de Saida (V)')
axes[2].set_title(f'Saida Amplificada (Ganho={Ganho_InAmp:.1f}x)')
axes[2].axhline(y=5.0, color='k', linestyle='--', alpha=0.5, label='Limite ADC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Condicionamento de Sinal - Sensor RTD PT100', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nResumo do Projeto:")
print(f"  PT100 @ 0C: {R0:.0f} ohms -> V_bridge = 0 mV")
print(f"  PT100 @ 200C: {R0*(1+alpha*200):.1f} ohms -> V_bridge = {V_bridge_max*1000:.2f} mV")
print(f"  Ganho necessario do InAmp: {Ganho_InAmp:.1f}x")

## Analise de Sinais: FFT e Filtragem

Nesta secao, vamos gerar um sinal ruidoso com componentes de frequencia
conhecidas e usar o Claude para nos ajudar a analisar o espectro e
projetar um filtro digital apropriado.

In [ ]:
# Gerar sinal de teste com multiplas componentes de frequencia + ruido
fs = 1000  # Frequencia de amostragem (Hz)
t = np.linspace(0, 1, fs, endpoint=False)  # 1 segundo de dados

# Sinal composto: 10Hz (sinal util) + 60Hz (ruido de rede) + 200Hz (harmonico)
sinal_10hz = 2.0 * np.sin(2 * np.pi * 10 * t)   # Sinal principal
ruido_60hz = 0.8 * np.sin(2 * np.pi * 60 * t)    # Interferencia da rede eletrica
harmonico_200hz = 0.3 * np.sin(2 * np.pi * 200 * t)  # Harmonico espurio
ruido_branco = 0.5 * np.random.randn(len(t))     # Ruido gaussiano

sinal_ruidoso = sinal_10hz + ruido_60hz + harmonico_200hz + ruido_branco

# Plot do sinal no dominio do tempo
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t[:200], sinal_10hz[:200], 'g--', alpha=0.7, label='Sinal puro (10 Hz)')
axes[0].plot(t[:200], sinal_ruidoso[:200], 'b-', alpha=0.8, label='Sinal ruidoso')
axes[0].set_xlabel('Tempo (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Sinal no Dominio do Tempo')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Calcular e plotar a FFT
N = len(sinal_ruidoso)
fft_result = np.fft.fft(sinal_ruidoso)
fft_magnitude = 2.0 / N * np.abs(fft_result[:N // 2])
freqs = np.fft.fftfreq(N, 1.0 / fs)[:N // 2]

axes[1].stem(freqs, fft_magnitude, linefmt='b-', markerfmt='bo', basefmt='k-', use_line_collection=True)
axes[1].set_xlabel('Frequencia (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Espectro de Frequencia (FFT)')
axes[1].set_xlim(0, 300)
axes[1].grid(True, alpha=0.3)

# Anotar picos
for freq, label in [(10, '10 Hz\n(sinal)'), (60, '60 Hz\n(rede)'), (200, '200 Hz\n(harmonico)')]:
    idx = np.argmin(np.abs(freqs - freq))
    axes[1].annotate(label, xy=(freqs[idx], fft_magnitude[idx]),
                     xytext=(freqs[idx]+15, fft_magnitude[idx]+0.1),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# Pedir ao Claude para analisar o sinal e recomendar filtros
prompt_analise = f"""
Analisei um sinal amostrado a {fs} Hz e encontrei as seguintes componentes
no espectro FFT:

- Pico em 10 Hz com magnitude ~2.0 (sinal util de um sensor)
- Pico em 60 Hz com magnitude ~0.8 (interferencia da rede eletrica)
- Pico em 200 Hz com magnitude ~0.3 (harmonico espurio)
- Ruido branco gaussiano distribuido em todo o espectro

Preciso extrair apenas o sinal de 10 Hz. Recomende:

1. Que tipo de filtro digital usar (FIR ou IIR? Que topologia?)
2. Parametros do filtro (ordem, frequencia de corte)
3. Codigo Python usando scipy.signal para implementar o filtro
4. Cuidados com atraso de grupo e estabilidade
"""

resposta_filtro = ask_claude(prompt_analise)
print(resposta_filtro)

In [ ]:
# Implementar filtro Butterworth passa-baixa baseado na analise
fc = 30  # Frequencia de corte em Hz (entre 10Hz e 60Hz)
ordem = 4

# Projetar filtro Butterworth
b, a = scipy_signal.butter(ordem, fc, btype='low', fs=fs)

# Aplicar filtro (filtfilt para zero atraso de fase)
sinal_filtrado = scipy_signal.filtfilt(b, a, sinal_ruidoso)

# Comparar sinais
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Sinal original vs filtrado no tempo
axes[0, 0].plot(t[:300], sinal_ruidoso[:300], 'b-', alpha=0.5, label='Ruidoso')
axes[0, 0].plot(t[:300], sinal_filtrado[:300], 'r-', linewidth=2, label='Filtrado')
axes[0, 0].plot(t[:300], sinal_10hz[:300], 'g--', alpha=0.7, label='Original (10 Hz)')
axes[0, 0].set_title('Comparacao no Dominio do Tempo')
axes[0, 0].set_xlabel('Tempo (s)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# FFT do sinal filtrado
fft_filtrado = np.fft.fft(sinal_filtrado)
fft_mag_filtrado = 2.0 / N * np.abs(fft_filtrado[:N // 2])

axes[0, 1].plot(freqs, fft_magnitude, 'b-', alpha=0.5, label='Antes do filtro')
axes[0, 1].plot(freqs, fft_mag_filtrado, 'r-', linewidth=2, label='Apos filtro')
axes[0, 1].set_xlim(0, 300)
axes[0, 1].set_title('Comparacao no Espectro')
axes[0, 1].set_xlabel('Frequencia (Hz)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Resposta em frequencia do filtro
w, h = scipy_signal.freqz(b, a, worN=2048, fs=fs)
axes[1, 0].plot(w, 20 * np.log10(np.abs(h)), 'k-', linewidth=2)
axes[1, 0].axvline(x=fc, color='r', linestyle='--', label=f'fc = {fc} Hz')
axes[1, 0].axhline(y=-3, color='g', linestyle='--', alpha=0.5, label='-3 dB')
axes[1, 0].set_title(f'Resposta em Frequencia - Butterworth Ordem {ordem}')
axes[1, 0].set_xlabel('Frequencia (Hz)')
axes[1, 0].set_ylabel('Magnitude (dB)')
axes[1, 0].set_xlim(0, 300)
axes[1, 0].set_ylim(-80, 5)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Erro entre sinal original e filtrado
erro = sinal_10hz - sinal_filtrado
axes[1, 1].plot(t, erro, 'purple', alpha=0.7)
axes[1, 1].set_title(f'Erro de Reconstrucao (RMS = {np.sqrt(np.mean(erro**2)):.4f})')
axes[1, 1].set_xlabel('Tempo (s)')
axes[1, 1].set_ylabel('Erro')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Filtragem Digital - Butterworth Passa-Baixa', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Firmware para Arduino

Vamos usar o Claude para gerar codigo C++ completo para Arduino,
incluindo leitura de sensor DHT22 (temperatura e umidade),
saida serial formatada e tratamento de erros.

In [ ]:
prompt_arduino = """
Gere um sketch completo para Arduino (C++) que faca o seguinte:

1. Leia um sensor DHT22 conectado ao pino digital 2
2. Leia um sensor analogico (ex: LDR ou potenciometro) no pino A0
3. Calcule a media movel das ultimas 10 leituras do sensor analogico
4. Envie os dados pela serial (9600 baud) em formato CSV:
   timestamp_ms, temperatura_C, umidade_pct, analogico_raw, analogico_media
5. Pisque o LED embutido (pino 13) a cada leitura bem-sucedida
6. Trate erros de leitura do DHT22 (valores NaN)
7. Intervalo entre leituras: 2 segundos

Inclua comentarios explicativos no codigo.
Use a biblioteca DHT.h (Adafruit).
"""

resposta_arduino = ask_claude(prompt_arduino)
print(resposta_arduino)

In [ ]:
# Salvar o sketch gerado em um arquivo .ino
prompt_sketch = """
Gere APENAS o codigo C++ (sem explicacao, sem markdown) de um sketch Arduino
para ler o sensor DHT22 no pino 2, sensor analogico no A0,
com media movel de 10 amostras, saida CSV pela serial a 9600 baud,
LED no pino 13, tratamento de erro, intervalo de 2 segundos.
Use a biblioteca DHT.h (Adafruit).
"""

sketch_code = ask_claude(prompt_sketch)

# Salvar em arquivo
output_dir = "output_firmware"
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "sensor_monitor.ino"), "w") as f:
    f.write(sketch_code)

print(f"Sketch salvo em {output_dir}/sensor_monitor.ino")
print(f"Tamanho: {len(sketch_code)} caracteres")

## ESP32: IoT e Comunicacao WiFi

O ESP32 e um microcontrolador poderoso com WiFi e Bluetooth integrados.
Vamos usar o Claude para gerar firmware que leia sensores e envie
dados via HTTP e MQTT para um servidor IoT.

In [ ]:
prompt_esp32 = """
Gere firmware completo para ESP32 (Arduino framework) que faca:

1. Conecte ao WiFi (SSID e senha configuraveis como constantes)
2. Leia sensor DHT22 no GPIO 4
3. Leia sensor analogico (ex: sensor de luminosidade) no GPIO 34 (ADC)
4. Envie dados via HTTP POST para um endpoint REST (JSON payload):
   {"device_id": "esp32_lab01", "temp": 25.3, "humidity": 60.1, "light": 512}
5. Tambem publique via MQTT no topico "lab/sensores/esp32_01"
6. Implemente reconexao automatica WiFi e MQTT
7. Use deep sleep entre leituras (configurable, ex: 30 segundos)
8. LED de status no GPIO 2

Inclua:
- Estrutura modular com funcoes separadas
- Tratamento de erros em cada etapa
- Comentarios explicando cada secao
- Bibliotecas necessarias: WiFi.h, HTTPClient.h, PubSubClient.h, DHT.h
"""

resposta_esp32 = ask_claude(prompt_esp32)
print(resposta_esp32)

In [ ]:
# Pedir ao Claude para explicar a arquitetura do firmware ESP32
prompt_arquitetura = """
Explique a arquitetura tipica de firmware para ESP32 IoT, incluindo:

1. Diagrama de estados (inicializacao -> leitura -> envio -> sleep)
2. Comparacao entre HTTP vs MQTT para IoT (latencia, overhead, confiabilidade)
3. Estrategias de economia de energia (deep sleep, light sleep, modem sleep)
4. Calculo de autonomia da bateria para diferentes modos de sleep
5. Boas praticas de seguranca (TLS, autenticacao MQTT)

Apresente de forma tecnica mas acessivel para estudantes de engenharia.
"""

resposta_arquitetura = ask_claude(prompt_arquitetura)
print(resposta_arquitetura)

## Raspberry Pi: Aquisicao de Dados com Python

O Raspberry Pi, combinado com um ADC externo como o MCP3008,
permite aquisicao de dados analogicos com Python. Vamos gerar
codigo para leitura de sensores e registro em CSV.

In [ ]:
prompt_raspi = """
Gere codigo Python completo para Raspberry Pi que faca aquisicao de dados
usando um ADC MCP3008 via SPI. Requisitos:

1. Leia 4 canais analogicos do MCP3008 (CH0 a CH3):
   - CH0: Sensor de temperatura (termistor NTC 10k)
   - CH1: Sensor de luminosidade (LDR)
   - CH2: Sensor de pressao (MPX4115)
   - CH3: Potenciometro de referencia

2. Converta os valores ADC (0-1023) para unidades fisicas:
   - Temperatura em Celsius (usando equacao de Steinhart-Hart)
   - Luminosidade em lux (aproximacao logaritmica)
   - Pressao em kPa

3. Registre dados em CSV com timestamp (colunas: timestamp, temp_C, lux, pressao_kPa, ref_V)

4. Implemente:
   - Taxa de amostragem configuravel (ex: 1 Hz)
   - Buffer circular em memoria antes de gravar no disco
   - Rotacao de arquivo CSV a cada hora
   - Tratamento de excecoes SPI e de arquivo
   - Sinal de parada gracioso (SIGINT/SIGTERM)

Use as bibliotecas: spidev, RPi.GPIO, csv, time, signal.
Inclua comentarios detalhados.
"""

resposta_raspi = ask_claude(prompt_raspi)
print(resposta_raspi)

In [ ]:
# Simular dados de aquisicao como o Raspberry Pi faria
# (podemos rodar isso em qualquer ambiente Python)

np.random.seed(42)
n_amostras = 500
tempo = np.arange(n_amostras)  # Segundos

# Simular leituras dos sensores
temp_simulada = 22 + 3 * np.sin(2 * np.pi * tempo / 300) + 0.2 * np.random.randn(n_amostras)
lux_simulado = 500 + 200 * np.sin(2 * np.pi * tempo / 600) + 30 * np.random.randn(n_amostras)
pressao_simulada = 101.3 + 0.5 * np.sin(2 * np.pi * tempo / 450) + 0.1 * np.random.randn(n_amostras)
ref_simulada = 3.3 + 0.01 * np.random.randn(n_amostras)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(tempo, temp_simulada, 'r-', linewidth=1)
axes[0].set_ylabel('Temperatura (C)')
axes[0].set_title('CH0: Termistor NTC 10k')
axes[0].grid(True, alpha=0.3)

axes[1].plot(tempo, lux_simulado, 'orange', linewidth=1)
axes[1].set_ylabel('Luminosidade (lux)')
axes[1].set_title('CH1: Sensor LDR')
axes[1].grid(True, alpha=0.3)

axes[2].plot(tempo, pressao_simulada, 'b-', linewidth=1)
axes[2].set_ylabel('Pressao (kPa)')
axes[2].set_title('CH2: Sensor MPX4115')
axes[2].grid(True, alpha=0.3)

axes[3].plot(tempo, ref_simulada, 'g-', linewidth=1)
axes[3].set_ylabel('Tensao (V)')
axes[3].set_xlabel('Tempo (s)')
axes[3].set_title('CH3: Referencia (Potenciometro)')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Simulacao de Aquisicao de Dados - Raspberry Pi + MCP3008', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Amostras coletadas: {n_amostras}")
print(f"Temperatura: media={np.mean(temp_simulada):.1f}C, std={np.std(temp_simulada):.2f}C")
print(f"Luminosidade: media={np.mean(lux_simulado):.0f} lux")
print(f"Pressao: media={np.mean(pressao_simulada):.1f} kPa")

## Calculando Filtros Analogicos

Vamos projetar um filtro Butterworth passa-baixa analogico,
calcular os valores dos componentes (resistores e capacitores),
e visualizar a funcao de transferencia e o diagrama de Bode.

In [ ]:
prompt_filtro_analogico = """
Projete um filtro Butterworth passa-baixa analogico com as especificacoes:

- Frequencia de corte (-3dB): 1 kHz
- Ordem: 4 (implementado como cascata de 2 secoes de 2a ordem Sallen-Key)
- Alimentacao: +/- 12V (com op-amps como TL072 ou LM358)

Faca:
1. Calcule os polos do filtro Butterworth de 4a ordem
2. Determine os fatores de qualidade Q de cada secao
3. Calcule valores de R e C para cada estagio Sallen-Key
   (assuma resistores iguais, calcule os capacitores)
4. Escreva a funcao de transferencia H(s) completa
5. Indique valores comerciais mais proximos (serie E24/E96)
"""

resposta_filtro_analogico = ask_claude(prompt_filtro_analogico)
print(resposta_filtro_analogico)

In [ ]:
# Projetar e visualizar o filtro Butterworth de 4a ordem
fc_analog = 1000  # Hz
wc = 2 * np.pi * fc_analog  # rad/s
ordem_analog = 4

# Criar filtro Butterworth analogico
b_analog, a_analog = scipy_signal.butter(ordem_analog, wc, btype='low', analog=True)

# Resposta em frequencia
w_range = np.logspace(1, 5, 1000)  # 10 Hz a 100 kHz
w_resp, h_resp = scipy_signal.freqs(b_analog, a_analog, worN=w_range)

# Polos e zeros
zeros, polos, ganho = scipy_signal.tf2zpk(b_analog, a_analog)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Diagrama de Bode - Magnitude
freq_hz = w_resp / (2 * np.pi)
mag_db = 20 * np.log10(np.abs(h_resp))
axes[0, 0].semilogx(freq_hz, mag_db, 'b-', linewidth=2)
axes[0, 0].axvline(x=fc_analog, color='r', linestyle='--', alpha=0.7, label=f'fc = {fc_analog} Hz')
axes[0, 0].axhline(y=-3, color='g', linestyle='--', alpha=0.5, label='-3 dB')
axes[0, 0].axhline(y=-80, color='gray', linestyle=':', alpha=0.3)
axes[0, 0].set_xlabel('Frequencia (Hz)')
axes[0, 0].set_ylabel('Magnitude (dB)')
axes[0, 0].set_title('Diagrama de Bode - Magnitude')
axes[0, 0].set_ylim(-100, 5)
axes[0, 0].legend()
axes[0, 0].grid(True, which='both', alpha=0.3)

# Diagrama de Bode - Fase
fase_graus = np.angle(h_resp, deg=True)
axes[0, 1].semilogx(freq_hz, fase_graus, 'r-', linewidth=2)
axes[0, 1].axvline(x=fc_analog, color='b', linestyle='--', alpha=0.7, label=f'fc = {fc_analog} Hz')
axes[0, 1].axhline(y=-180, color='gray', linestyle=':', alpha=0.3)
axes[0, 1].set_xlabel('Frequencia (Hz)')
axes[0, 1].set_ylabel('Fase (graus)')
axes[0, 1].set_title('Diagrama de Bode - Fase')
axes[0, 1].legend()
axes[0, 1].grid(True, which='both', alpha=0.3)

# Diagrama de Polos e Zeros
theta = np.linspace(0, 2*np.pi, 100)
axes[1, 0].plot(wc * np.cos(theta), wc * np.sin(theta), 'k--', alpha=0.3, label=f'|s| = wc')
axes[1, 0].scatter(np.real(polos), np.imag(polos), marker='x', s=100, c='red', linewidths=2, label='Polos')
if len(zeros) > 0:
    axes[1, 0].scatter(np.real(zeros), np.imag(zeros), marker='o', s=100, c='blue', linewidths=2, label='Zeros')
axes[1, 0].axhline(y=0, color='k', linewidth=0.5)
axes[1, 0].axvline(x=0, color='k', linewidth=0.5)
axes[1, 0].set_xlabel('Real (rad/s)')
axes[1, 0].set_ylabel('Imaginario (rad/s)')
axes[1, 0].set_title('Diagrama de Polos e Zeros')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_aspect('equal')

# Resposta ao degrau
t_step = np.linspace(0, 0.005, 1000)  # 5 ms
sys_analog = scipy_signal.TransferFunction(b_analog, a_analog)
t_out, y_out = scipy_signal.step(sys_analog, T=t_step)
axes[1, 1].plot(t_out * 1000, y_out, 'purple', linewidth=2)
axes[1, 1].axhline(y=1.0, color='k', linestyle='--', alpha=0.5, label='Regime permanente')
axes[1, 1].set_xlabel('Tempo (ms)')
axes[1, 1].set_ylabel('Amplitude')
axes[1, 1].set_title('Resposta ao Degrau Unitario')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Filtro Butterworth Passa-Baixa - Ordem {ordem_analog}, fc = {fc_analog} Hz',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Imprimir informacoes dos polos
print("\nPolos do filtro:")
for i, p in enumerate(polos):
    freq_polo = np.abs(p) / (2 * np.pi)
    Q = np.abs(p) / (-2 * np.real(p)) if np.real(p) != 0 else float('inf')
    print(f"  Polo {i+1}: {p:.1f} rad/s | f = {freq_polo:.1f} Hz | Q = {Q:.3f}")

In [ ]:
# Comparar filtros de diferentes ordens
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

cores = ['blue', 'green', 'red', 'purple', 'orange']
for i, n in enumerate([1, 2, 3, 4, 6]):
    b_n, a_n = scipy_signal.butter(n, wc, btype='low', analog=True)
    w_n, h_n = scipy_signal.freqs(b_n, a_n, worN=w_range)
    ax.semilogx(w_n / (2*np.pi), 20*np.log10(np.abs(h_n)),
                color=cores[i], linewidth=2, label=f'Ordem {n} (-{20*n} dB/dec)')

ax.axvline(x=fc_analog, color='black', linestyle='--', alpha=0.5)
ax.axhline(y=-3, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequencia (Hz)', fontsize=12)
ax.set_ylabel('Magnitude (dB)', fontsize=12)
ax.set_title('Comparacao de Filtros Butterworth de Diferentes Ordens', fontsize=14)
ax.set_ylim(-100, 5)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
ax.annotate(f'fc = {fc_analog} Hz', xy=(fc_analog, -3), xytext=(fc_analog*3, 2),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=11)

plt.tight_layout()
plt.show()

## Exercicios Propostos

### Exercicio 1: Circuito de Condicionamento
Modifique o circuito da ponte de Wheatstone para usar um termopar tipo K
em vez de um PT100. Peca ao Claude para calcular os novos valores de
componentes considerando que o termopar gera ~41 uV/C.

### Exercicio 2: Filtro Notch
Projete um filtro notch (rejeita-faixa) digital para eliminar
especificamente a interferencia de 60 Hz do sinal, preservando
tanto o sinal de 10 Hz quanto o harmonico de 200 Hz.

### Exercicio 3: Arduino + PID
Peca ao Claude para gerar um controlador PID em Arduino que
mantenha a temperatura de um forno em um setpoint desejado,
usando um sensor PT100 e um rele de estado solido (SSR).

### Exercicio 4: ESP32 + Bluetooth
Modifique o firmware do ESP32 para enviar dados via Bluetooth Low Energy
(BLE) em vez de WiFi. Peca ao Claude para implementar um GATT server
com caracteristicas para temperatura, umidade e luminosidade.

### Exercicio 5: Sistema de Aquisicao Completo
Combine o Raspberry Pi com o MCP3008 para criar um sistema de aquisicao
que detecte automaticamente anomalias nos dados usando um limiar
estatistico (media +/- 3 sigma) e envie alertas via email.

### Exercicio 6: Filtro Chebyshev
Repita o projeto do filtro analogico usando topologia Chebyshev Tipo I
com 0.5 dB de ripple na banda passante. Compare o diagrama de Bode
com o Butterworth equivalente e discuta as diferencas.

## Proximos Passos

1. **Aprofundar em PCB Design**: Use o Claude para ajudar com layout de
   placas de circuito impresso, regras de roteamento e design for manufacturing (DFM)

2. **Processamento de Sinais Avancado**: Explore wavelets, filtros adaptativos
   e analise tempo-frequencia com auxilio do Claude

3. **RTOS para Embarcados**: Aprenda a usar FreeRTOS no ESP32 para
   multitarefa em tempo real (leitura de sensores + comunicacao simultanea)

4. **Protocolos Industriais**: Explore comunicacao Modbus RTU/TCP,
   CAN bus e protocolo HART para instrumentacao industrial

5. **Machine Learning em Edge**: Implemente modelos TensorFlow Lite no
   ESP32 ou Raspberry Pi para deteccao de anomalias em tempo real

6. **Simulacao SPICE**: Use o Claude para gerar netlists SPICE e analisar
   circuitos analogicos complexos antes da prototipagem

---

**Nota:** Os codigos de firmware gerados pelo Claude devem ser revisados e
testados antes de uso em hardware real. Sempre verifique limites de tensao,
corrente e compatibilidade de pinos antes de conectar componentes.